### **Semana 6- Herramientas (tools), llamada de funciones (function calling), contratos, validación, errores, reintentos (retries) y límites de tiempo (timeouts)**

### **Pregunta experimental**

> ¿Cómo cambia la correctitud y robustez de la ejecución cuando se incorporan progresivamente contratos tipados, errores estructurados, retries selectivos y límites temporales, manteniendo constantes herramientas, entradas y fallos?.

```text
LLM probabilístico
      |
      v
`ToolCall` (llamada a herramienta)
      |
      v
software determinista
```

Esta semana no implementa agentes, ReAct, planificación, LangGraph ni MCP.

In [ ]:
from __future__ import annotations

import asyncio
import hashlib
import json
import os
import platform
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Literal

import pandas as pd
import pydantic
from pydantic import BaseModel, Field, ValidationError

SEED = 20261005
RUN_REAL_TOOL_LLM = os.getenv("CC0F4_RUN_REAL_TOOL_LLM", "0") == "1"
REAL_MODEL_ID = os.getenv("CC0F4_TOOL_MODEL_ID", "Qwen/Qwen3-1.7B")

WEEK_DIR = Path("Semana6") if Path("Semana6").exists() else Path(".")
DATA_DIR = WEEK_DIR / "datos"
RESULTS_DIR = WEEK_DIR / "resultados"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Pydantic:", pydantic.__version__)
print("Modo LLM real:", RUN_REAL_TOOL_LLM)


#### **1. Cargar benchmark y catálogo**

El fault schedule es dato experimental.

No se modifica entre condiciones A/B/C/D/E.

In [ ]:
catalog = json.loads((DATA_DIR / "catalogo_tools.json").read_text(encoding="utf-8"))

runtime_cases = [
    json.loads(line)
    for line in (DATA_DIR / "benchmark_runtime.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

llm_cases = [
    json.loads(line)
    for line in (DATA_DIR / "benchmark_tool_calls.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

fixtures = {
    item["id"]: item["raw"]
    for item in json.loads((DATA_DIR / "tool_call_fixtures.json").read_text(encoding="utf-8"))
}

assert len(runtime_cases) == 17
assert len(llm_cases) == 24

print("Runtime cases:", len(runtime_cases))
print("Tool-call cases:", len(llm_cases))


#### **2. Contratos Pydantic**

```text
contrato de entrada (`input contract`) != contrato de dominio (`domain contract`) != contrato de salida (`output contract`)
```

In [ ]:
class CalculateTotalInput(BaseModel):
    unit_price: float = Field(gt=0)
    quantity: int = Field(ge=1, le=100)
    discount_pct: float = Field(default=0.0, ge=0, le=100)


class LookupStockInput(BaseModel):
    sku: str = Field(min_length=3, max_length=20)


class ShippingQuoteInput(BaseModel):
    origin: str = Field(min_length=3, max_length=3)
    destination: str = Field(min_length=3, max_length=3)
    weight_kg: float = Field(gt=0, le=50)


class CalculateTotalOutput(BaseModel):
    subtotal: float
    discount: float
    total: float


class LookupStockOutput(BaseModel):
    sku: str
    available: int


class ShippingQuoteOutput(BaseModel):
    origin: str
    destination: str
    weight_kg: float
    price: float


INPUT_MODELS = {
    "calculator.calculate_total": CalculateTotalInput,
    "catalog.lookup_stock": LookupStockInput,
    "shipping.get_quote": ShippingQuoteInput,
}

OUTPUT_MODELS = {
    "calculator.calculate_total": CalculateTotalOutput,
    "catalog.lookup_stock": LookupStockOutput,
    "shipping.get_quote": ShippingQuoteOutput,
}


#### **3. Error como contrato**

```text
exception != ToolError
```

In [ ]:
ErrorKind = Literal[
    "protocol",
    "validation",
    "domain",
    "transient",
    "timeout",
    "execution",
    "output_contract",
]


class ToolError(BaseModel):
    code: str
    kind: ErrorKind
    retryable: bool
    message: str


class ToolResult(BaseModel):
    status: Literal["ok", "error"]
    data: dict[str, Any] | None = None
    error: ToolError | None = None


def error_result(
    kind: ErrorKind,
    code: str,
    message: str,
    retryable: bool,
) -> ToolResult:
    return ToolResult(
        status="error",
        error=ToolError(
            code=code,
            kind=kind,
            retryable=retryable,
            message=message,
        ),
    )


#### **4. Herramientas deterministas e inyección de fallos (fault injection)**

```text
fault
```

es un parámetro del experimento.

En producción no formaría parte de la interfaz pública.

In [ ]:
class DomainError(Exception):
    pass


class TransientToolError(Exception):
    pass


async def calculate_total(
    args: dict[str, Any],
    fault: str,
    attempt: int,
) -> dict[str, Any]:
    subtotal = float(args["unit_price"]) * int(args["quantity"])
    discount = subtotal * float(args.get("discount_pct", 0.0)) / 100.0
    return {
        "subtotal": round(subtotal, 2),
        "discount": round(discount, 2),
        "total": round(subtotal - discount, 2),
    }


async def lookup_stock(
    args: dict[str, Any],
    fault: str,
    attempt: int,
) -> dict[str, Any]:
    sku = str(args["sku"])
    if sku not in catalog["stock"]:
        raise DomainError(f"SKU inexistente: {sku}")
    return {
        "sku": sku,
        "available": int(catalog["stock"][sku]),
    }


async def shipping_quote(
    args: dict[str, Any],
    fault: str,
    attempt: int,
) -> dict[str, Any]:
    origin = str(args["origin"]).upper()
    destination = str(args["destination"]).upper()
    route = f"{origin}-{destination}"

    if route not in catalog["routes"]:
        raise DomainError(f"Ruta no disponible: {route}")

    if fault == "transient_once" and attempt <= 1:
        raise TransientToolError("Fallo transitorio inyectado")

    if fault == "transient_twice" and attempt <= 2:
        raise TransientToolError("Fallo transitorio inyectado")

    if fault == "always_transient":
        raise TransientToolError("Fallo transitorio persistente")

    if fault == "slow_once" and attempt <= 1:
        await asyncio.sleep(0.14)

    if fault == "always_slow":
        await asyncio.sleep(0.14)

    if fault == "internal_error":
        raise RuntimeError("Excepción interna inyectada")

    weight = float(args["weight_kg"])
    price = float(catalog["routes"][route]) + 1.25 * weight

    result = {
        "origin": origin,
        "destination": destination,
        "weight_kg": weight,
        "price": round(price, 2),
    }

    if fault == "malformed_output":
        return {
            "route": route,
            "cost": "unknown",
        }

    return result


TOOL_IMPLS = {
    "calculator.calculate_total": calculate_total,
    "catalog.lookup_stock": lookup_stock,
    "shipping.get_quote": shipping_quote,
}


#### **5. Políticas A/B/C/D/E**

Cada condición agrega una sola familia principal de mecanismos.

In [ ]:
@dataclass(frozen=True)
class Policy:
    condition: str
    validate_input: bool
    structured_errors: bool
    retry_transient: bool
    use_timeout: bool
    max_attempts: int = 3
    backoff_ms: tuple[int, ...] = (20, 40)
    per_attempt_timeout_ms: int = 100
    overall_deadline_ms: int = 230


POLICIES = [
    Policy("A", False, False, False, False, max_attempts=1),
    Policy("B", True, False, False, False, max_attempts=1),
    Policy("C", True, True, False, False, max_attempts=1),
    Policy("D", True, True, True, False, max_attempts=3),
    Policy("E", True, True, True, True, max_attempts=3),
]

POLICY_BY_CONDITION = {
    policy.condition: policy
    for policy in POLICIES
}


#### **6. Entorno de ejecución experimental (runtime)**

Puntos críticos:

```text
herramienta desconocida (`herramienta desconocida (`unknown tool`)`)
-> protocolo (`protocol`)

Pydantic failure
-> validación (`validation`)

DomainError
-> domain

TransientToolError
-> transitorio (`transient`)

asyncio timeout
-> timeout

unexpected exception
-> ejecución (`execution`)

output Pydantic failure
-> contrato de salida (`output_contract`)
```

Antes de realizar la espera progresiva (`backoff`) en E se verifica que el límite temporal total (`deadline`) permita continuar.

In [ ]:
async def run_case(
    case: dict[str, Any],
    policy: Policy,
) -> dict[str, Any]:
    started = time.perf_counter()
    tool_name = case["tool"]
    raw_args = dict(case["arguments"])
    fault = case["fault"]

    traces: list[dict[str, Any]] = []
    invalid_execution = False
    uncaught = False

    if tool_name not in TOOL_IMPLS:
        result = error_result(
            "protocol",
            "unknown_tool",
            f"Tool no permitida: {tool_name}",
            False,
        )
        return {
            "case_id": case["id"],
            "condition": policy.condition,
            "tool_name": tool_name,
            "result": result.model_dump(),
            "attempts": 0,
            "latency_ms": (time.perf_counter() - started) * 1000,
            "invalid_execution": False,
            "uncaught_exception": False,
            "traces": traces,
        }

    args = raw_args

    if policy.validate_input:
        try:
            args = INPUT_MODELS[tool_name].model_validate(raw_args).model_dump()
        except ValidationError as exc:
            result = error_result(
                "validation",
                "invalid_arguments",
                str(exc).splitlines()[0],
                False,
            )
            return {
                "case_id": case["id"],
                "condition": policy.condition,
                "tool_name": tool_name,
                "result": result.model_dump(),
                "attempts": 0,
                "latency_ms": (time.perf_counter() - started) * 1000,
                "invalid_execution": False,
                "uncaught_exception": False,
                "traces": traces,
            }

    attempt = 0

    while True:
        attempt += 1
        attempt_started = time.perf_counter()

        # Solo para medir la condición A:
        # ¿llegó a ejecutarse un input que el contrato habría rechazado?
        if not policy.validate_input:
            try:
                INPUT_MODELS[tool_name].model_validate(raw_args)
            except ValidationError:
                invalid_execution = True

        try:
            coro = TOOL_IMPLS[tool_name](args, fault, attempt)

            if policy.use_timeout:
                elapsed_ms = (time.perf_counter() - started) * 1000
                remaining_ms = policy.overall_deadline_ms - elapsed_ms

                if remaining_ms <= 0:
                    result = error_result(
                        "timeout",
                        "deadline_exceeded",
                        "Deadline total agotado antes del intento.",
                        True,
                    )
                    break

                timeout_s = min(
                    policy.per_attempt_timeout_ms,
                    remaining_ms,
                ) / 1000.0

                data = await asyncio.wait_for(
                    coro,
                    timeout=timeout_s,
                )
            else:
                data = await coro

            if policy.structured_errors:
                try:
                    data = (
                        OUTPUT_MODELS[tool_name]
                        .model_validate(data)
                        .model_dump()
                    )
                except ValidationError:
                    traces.append({
                        "attempt": attempt,
                        "status": "error",
                        "error_kind": "output_contract",
                        "retryable": False,
                        "latency_ms": (time.perf_counter() - attempt_started) * 1000,
                    })
                    result = error_result(
                        "output_contract",
                        "invalid_tool_output",
                        "La salida viola el contrato.",
                        False,
                    )
                    break

            result = ToolResult(
                status="ok",
                data=data,
            )
            traces.append({
                "attempt": attempt,
                "status": "ok",
                "error_kind": None,
                "retryable": False,
                "latency_ms": (time.perf_counter() - attempt_started) * 1000,
            })
            break

        except asyncio.TimeoutError:
            traces.append({
                "attempt": attempt,
                "status": "error",
                "error_kind": "timeout",
                "retryable": True,
                "latency_ms": (time.perf_counter() - attempt_started) * 1000,
            })

            if policy.retry_transient and attempt < policy.max_attempts:
                sleep_ms = (
                    policy.backoff_ms[attempt - 1]
                    if attempt - 1 < len(policy.backoff_ms)
                    else 0
                )

                if policy.use_timeout:
                    elapsed_ms = (time.perf_counter() - started) * 1000
                    remaining_ms = policy.overall_deadline_ms - elapsed_ms

                    if remaining_ms <= sleep_ms:
                        result = error_result(
                            "timeout",
                            "deadline_exceeded",
                            "No queda presupuesto para backoff y otro intento.",
                            True,
                        )
                        break

                if sleep_ms > 0:
                    await asyncio.sleep(sleep_ms / 1000.0)

                continue

            result = error_result(
                "timeout",
                "attempt_timeout",
                "Timeout por intento.",
                True,
            )
            break

        except TransientToolError as exc:
            traces.append({
                "attempt": attempt,
                "status": "error",
                "error_kind": "transient",
                "retryable": True,
                "latency_ms": (time.perf_counter() - attempt_started) * 1000,
            })

            if policy.retry_transient and attempt < policy.max_attempts:
                sleep_ms = (
                    policy.backoff_ms[attempt - 1]
                    if attempt - 1 < len(policy.backoff_ms)
                    else 0
                )

                if policy.use_timeout:
                    elapsed_ms = (time.perf_counter() - started) * 1000
                    remaining_ms = policy.overall_deadline_ms - elapsed_ms

                    if remaining_ms <= sleep_ms:
                        result = error_result(
                            "timeout",
                            "deadline_exceeded",
                            "No queda presupuesto para backoff y otro intento.",
                            True,
                        )
                        break

                if sleep_ms > 0:
                    await asyncio.sleep(sleep_ms / 1000.0)

                continue

            if policy.structured_errors:
                result = error_result(
                    "transient",
                    "temporary_unavailable",
                    str(exc),
                    True,
                )
            else:
                uncaught = True
                result = error_result(
                    "execution",
                    "uncaught_exception",
                    type(exc).__name__,
                    False,
                )
            break

        except DomainError as exc:
            traces.append({
                "attempt": attempt,
                "status": "error",
                "error_kind": "domain",
                "retryable": False,
                "latency_ms": (time.perf_counter() - attempt_started) * 1000,
            })

            if policy.structured_errors:
                result = error_result(
                    "domain",
                    "domain_error",
                    str(exc),
                    False,
                )
            else:
                uncaught = True
                result = error_result(
                    "execution",
                    "uncaught_exception",
                    type(exc).__name__,
                    False,
                )
            break

        except (KeyError, TypeError, ValueError) as exc:
            traces.append({
                "attempt": attempt,
                "status": "error",
                "error_kind": "validation",
                "retryable": False,
                "latency_ms": (time.perf_counter() - attempt_started) * 1000,
            })

            uncaught = True
            result = error_result(
                "execution",
                "uncaught_exception",
                type(exc).__name__,
                False,
            )
            break

        except Exception as exc:
            traces.append({
                "attempt": attempt,
                "status": "error",
                "error_kind": "execution",
                "retryable": False,
                "latency_ms": (time.perf_counter() - attempt_started) * 1000,
            })

            if policy.structured_errors:
                result = error_result(
                    "execution",
                    "tool_execution_error",
                    str(exc),
                    False,
                )
            else:
                uncaught = True
                result = error_result(
                    "execution",
                    "uncaught_exception",
                    type(exc).__name__,
                    False,
                )
            break

    return {
        "case_id": case["id"],
        "condition": policy.condition,
        "tool_name": tool_name,
        "result": result.model_dump(),
        "attempts": attempt,
        "latency_ms": (time.perf_counter() - started) * 1000,
        "invalid_execution": invalid_execution,
        "uncaught_exception": uncaught,
        "traces": traces,
    }


#### **7. Ejecutar A/B/C/D/E**

In [ ]:
all_runs = []

for policy in POLICIES:
    for case in runtime_cases:
        all_runs.append(
            await run_case(case, policy)
        )

assert len(all_runs) == len(POLICIES) * len(runtime_cases)
print("Ejecuciones:", len(all_runs))


#### **8. Outcomes y clasificación**

Un error esperado puede ser un outcome correcto.

In [ ]:
expected_by_id = {
    case["id"]: case["expected"]
    for case in runtime_cases
}

fault_by_id = {
    case["id"]: case["fault"]
    for case in runtime_cases
}


def observed_kind(run: dict[str, Any]) -> str | None:
    if run["result"]["status"] == "ok":
        return None
    error = run["result"].get("error") or {}
    return error.get("kind")


rows = []

for run in all_runs:
    expected = expected_by_id[run["case_id"]]
    actual_status = run["result"]["status"]
    actual_kind = observed_kind(run)

    expected_status = expected["status"]
    expected_kind = expected.get("kind")

    correct = (
        actual_status == expected_status
        and (
            expected_status == "ok"
            or actual_kind == expected_kind
        )
    )

    classification_correct = (
        True
        if expected_status == "ok"
        else actual_kind == expected_kind
    )

    rows.append({
        "case_id": run["case_id"],
        "condition": run["condition"],
        "tool_name": run["tool_name"],
        "fault": fault_by_id[run["case_id"]],
        "actual_status": actual_status,
        "actual_kind": actual_kind,
        "expected_status": expected_status,
        "expected_kind": expected_kind,
        "correct_outcome": correct,
        "classification_correct": classification_correct,
        "attempts": run["attempts"],
        "latency_ms": run["latency_ms"],
        "invalid_execution": run["invalid_execution"],
        "uncaught_exception": run["uncaught_exception"],
    })

df = pd.DataFrame(rows)
df.head()


#### **9. Métricas del Experimento A**

Corrección respecto a una versión previa:

```text
`deadline_overrun_rate` (tasa de superación del límite temporal)
```

se calcula a partir de:

```python
policy.use_timeout
```

y no de un nombre hardcodeado como `condition == "E"`.

In [ ]:
def p95(values) -> float:
    values = sorted(float(x) for x in values)
    if not values:
        return 0.0

    index = int(round(0.95 * (len(values) - 1)))
    index = max(0, min(index, len(values) - 1))
    return values[index]


metrics_a = []

for condition, group in df.groupby("condition", sort=True):
    policy = POLICY_BY_CONDITION[condition]

    transient_group = group[
        group["fault"].isin([
            "transient_once",
            "transient_twice",
        ])
    ]

    if policy.use_timeout:
        # Tolerancia mínima para overhead del scheduler/medición.
        deadline_limit_ms = policy.overall_deadline_ms + 5.0
        deadline_overrun_rate = float(
            (group["latency_ms"] > deadline_limit_ms).mean()
        )
    else:
        deadline_overrun_rate = 0.0

    error_rows = group[
        group["expected_status"] == "error"
    ]

    metrics_a.append({
        "condition": condition,
        "correct_outcome_rate": float(group["correct_outcome"].mean()),
        "uncaught_exception_rate": float(group["uncaught_exception"].mean()),
        "error_classification_accuracy": float(
            error_rows["classification_correct"].mean()
            if len(error_rows)
            else 1.0
        ),
        "invalid_execution_rate": float(group["invalid_execution"].mean()),
        "transient_recovery_rate": float(
            (transient_group["actual_status"] == "ok").mean()
            if len(transient_group)
            else 0.0
        ),
        "mean_attempts": float(group["attempts"].mean()),
        "latency_mean_ms": float(group["latency_ms"].mean()),
        "latency_p95_ms": float(p95(group["latency_ms"])),
        "deadline_overrun_rate": deadline_overrun_rate,
    })


metrics_a_df = pd.DataFrame(metrics_a)
metrics_a_df


#### **10. Lectura causal**

```text
A vs B
-> `invalid_execution_rate` (tasa de ejecuciones inválidas)

B vs C
-> uncaught_exception_rate
-> error_classification_accuracy

C vs D
-> transitorio (`transient`)_recovery_rate
-> mean_attempts

D vs E
-> tiempo de espera (`timeout`)/límite temporal (`deadline`)
-> latency tail
```

In [ ]:
failures_a = (
    df[~df["correct_outcome"]]
    .sort_values(["condition", "case_id"])
)

failures_a[
    [
        "condition",
        "case_id",
        "fault",
        "expected_status",
        "expected_kind",
        "actual_status",
        "actual_kind",
        "attempts",
        "latency_ms",
    ]
]


#### **11. Trazas de herramientas (tool traces)**

Una métrica debe poder auditarse hasta un intento individual.

In [ ]:
trace_rows = []

for run in all_runs:
    for trace in run["traces"]:
        trace_rows.append({
            "case_id": run["case_id"],
            "condition": run["condition"],
            "tool_name": run["tool_name"],
            **trace,
        })

traces_df = pd.DataFrame(trace_rows)
traces_df.head(15)


### **Experimento B - Llamada de funciones (function calling)**

Ahora se fija el entorno de ejecución (`runtime`) y se mide la interfaz probabilística.

```text
prompt
-> 0 o 1 `ToolCall` (llamada a herramienta)
```

No hay loops.

In [ ]:
def normalize_tool_call(raw: Any) -> dict[str, Any] | None:
    if raw is None:
        return None

    if not isinstance(raw, dict):
        raise ValueError("ToolCall debe ser dict o None.")

    # Aceptar el envelope estándar:
    # {"type": "function", "function": {"name": ..., "arguments": ...}}
    if "function" in raw and isinstance(raw["function"], dict):
        raw = raw["function"]

    if "name" not in raw:
        raise ValueError("Falta name.")

    arguments = raw.get("arguments", {})

    if isinstance(arguments, str):
        arguments = json.loads(arguments)

    if not isinstance(arguments, dict):
        raise ValueError("arguments debe ser un objeto.")

    return {
        "name": str(raw["name"]),
        "arguments": arguments,
    }


def semantic_arguments_match(
    expected: dict[str, Any],
    actual: dict[str, Any],
) -> bool:
    expected = dict(expected)
    actual = dict(actual)

    if expected.get("discount_pct") == 0 and "discount_pct" not in actual:
        expected.pop("discount_pct")

    if actual.get("discount_pct") == 0 and "discount_pct" not in expected:
        actual.pop("discount_pct")

    if set(expected) != set(actual):
        return False

    for key, expected_value in expected.items():
        actual_value = actual[key]

        if (
            isinstance(expected_value, (int, float))
            and isinstance(actual_value, (int, float))
        ):
            if abs(float(expected_value) - float(actual_value)) > 1e-6:
                return False
        else:
            if str(expected_value).upper() != str(actual_value).upper():
                return False

    return True


#### **12. Especificaciones de herramientas (tool specifications) para Transformers**

In [ ]:
def tool_specs_for_transformers() -> list[dict[str, Any]]:
    return [
        {
            "type": "function",
            "function": {
                "name": name,
                "description": spec["description"],
                "parameters": spec["input_schema"],
            },
        }
        for name, spec in catalog["tools"].items()
    ]


TOOLS_FOR_TRANSFORMERS = tool_specs_for_transformers()

len(TOOLS_FOR_TRANSFORMERS)


#### **13. Parser robusto y explícito**

Primero se intenta el parser oficial del tokenizer si existe.

Fallback:

```text
`<tool_call>`
{...}
`</tool_call>`
```

No se realiza reparación automática (`auto-repair`).

In [ ]:
TOOL_CALL_BLOCK = re.compile(
    r"<tool_call>\s*(.*?)\s*</tool_call>",
    flags=re.DOTALL,
)


def extract_tool_calls_from_parsed_response(
    parsed: Any,
) -> list[dict[str, Any]]:
    if not isinstance(parsed, dict):
        return []

    calls = parsed.get("tool_calls") or []
    normalized = []

    for call in calls:
        normalized_call = normalize_tool_call(call)
        if normalized_call is not None:
            normalized.append(normalized_call)

    return normalized


def fallback_parse_tool_calls_from_text(
    text: str,
) -> list[dict[str, Any]]:
    text = text.strip()

    # Caso explícito: respuesta completa es JSON de ToolCall.
    try:
        obj = json.loads(text)
        call = normalize_tool_call(obj)
        return [] if call is None else [call]
    except Exception:
        pass

    # Caso habitual con wrapper <tool_call>.
    blocks = TOOL_CALL_BLOCK.findall(text)

    calls = []
    for block in blocks:
        block = block.strip()

        # La variante de la semana espera JSON dentro del bloque.
        # Si el modelo usa otra sintaxis, parse_response es la vía preferida.
        obj = json.loads(block)
        call = normalize_tool_call(obj)
        if call is not None:
            calls.append(call)

    return calls


def parse_generated_tool_call(
    tokenizer,
    raw_text: str,
    prefix_ids,
    tools: list[dict[str, Any]],
) -> dict[str, Any] | None:
    parsed_calls: list[dict[str, Any]] = []

    if hasattr(tokenizer, "parse_response"):
        try:
            parsed = tokenizer.parse_response(
                raw_text,
                prefix=prefix_ids,
                tools=tools,
            )
            parsed_calls = extract_tool_calls_from_parsed_response(parsed)
        except TypeError:
            # Compatibilidad con implementaciones donde parse_response
            # todavía no acepta `tools`.
            try:
                parsed = tokenizer.parse_response(
                    raw_text,
                    prefix=prefix_ids,
                )
                parsed_calls = extract_tool_calls_from_parsed_response(parsed)
            except Exception:
                parsed_calls = []
        except Exception:
            parsed_calls = []

    if not parsed_calls:
        parsed_calls = fallback_parse_tool_calls_from_text(raw_text)

    if len(parsed_calls) > 1:
        raise ValueError(
            "Semana 6 permite como máximo una tool call por prompt."
        )

    return parsed_calls[0] if parsed_calls else None


#### **14. Fixtures offline**

```text
fixtures -> validan software

fixtures != evaluación del modelo
```

In [ ]:
def get_offline_tool_call(
    case: dict[str, Any],
) -> dict[str, Any] | None:
    return fixtures[case["id"]]


#### **15. Modelo real opcional**

Se evita `Qwen-Agent`.

Se usa Transformers directamente.

El notebook intenta desactivar thinking mediante `enable_thinking=False` en el chat template. Si el template no acepta ese argumento, aplica la plantilla sin esa opción.

In [ ]:
real_tokenizer = None
real_model = None

if RUN_REAL_TOOL_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    real_tokenizer = AutoTokenizer.from_pretrained(
        REAL_MODEL_ID,
    )

    real_model = AutoModelForCausalLM.from_pretrained(
        REAL_MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
    )

    print("Modelo cargado:", REAL_MODEL_ID)
else:
    print("LLM real desactivado, se usarán fixtures.")


In [ ]:
def get_real_tool_call(
    case: dict[str, Any],
) -> dict[str, Any] | None:
    if not RUN_REAL_TOOL_LLM:
        return get_offline_tool_call(case)

    messages = [
        {
            "role": "system",
            "content": (
                "Decide si necesitas una herramienta. "
                "Realiza como máximo una llamada. "
                "Si no necesitas una herramienta, responde normalmente."
            ),
        },
        {
            "role": "user",
            "content": case["prompt"],
        },
    ]

    template_kwargs = dict(
        messages=messages,
        tools=TOOLS_FOR_TRANSFORMERS,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    try:
        inputs = real_tokenizer.apply_chat_template(
            **template_kwargs,
            enable_thinking=False,
        )
    except TypeError:
        inputs = real_tokenizer.apply_chat_template(
            **template_kwargs,
        )

    inputs = inputs.to(real_model.device)

    generated = real_model.generate(
        **inputs,
        max_new_tokens=160,
        do_sample=False,
    )

    new_tokens = generated[
        0,
        inputs["input_ids"].shape[1]:,
    ]

    raw_text = real_tokenizer.decode(
        new_tokens,
        skip_special_tokens=False,
    )

    return parse_generated_tool_call(
        tokenizer=real_tokenizer,
        raw_text=raw_text,
        prefix_ids=inputs["input_ids"][0],
        tools=TOOLS_FOR_TRANSFORMERS,
    )


#### **16. Métricas de llamada de funciones (function calling)**

Corrección importante:

Si:

```text
should_call_tool = True
```

pero:

```text
call = None
```

entonces:

```text
arguments_parse_ok = False
```

No se cuenta como análisis sintáctico (`parsing`) exitoso.

In [ ]:
llm_eval_rows = []

for case in llm_cases:
    call = None
    parser_exception = None

    try:
        call = normalize_tool_call(
            get_real_tool_call(case)
        )
    except Exception as exc:
        parser_exception = type(exc).__name__
        call = None

    should_call = bool(case["should_call_tool"])
    expected_tool = case["expected_tool"]

    selected_tool = (
        None
        if call is None
        else call["name"]
    )

    arguments = (
        {}
        if call is None
        else call["arguments"]
    )

    # Solo tiene sentido hablar de argumentos parseados
    # si el caso requería tool call.
    arguments_parse_ok = (
        call is not None and parser_exception is None
        if should_call
        else parser_exception is None
    )

    tool_choice_correct = (
        selected_tool == expected_tool
        if should_call
        else selected_tool is None
    )

    schema_valid = None
    semantic_ok = None

    if should_call:
        schema_valid = False
        semantic_ok = False

        if call is not None and selected_tool in INPUT_MODELS:
            try:
                validated = (
                    INPUT_MODELS[selected_tool]
                    .model_validate(arguments)
                    .model_dump(exclude_defaults=True)
                )
                schema_valid = True

                if selected_tool == expected_tool:
                    semantic_ok = semantic_arguments_match(
                        case["expected_arguments"],
                        validated,
                    )

            except ValidationError:
                schema_valid = False
                semantic_ok = False

    end_to_end = (
        tool_choice_correct
        and (
            not should_call
            or (
                arguments_parse_ok
                and bool(schema_valid)
                and bool(semantic_ok)
            )
        )
    )

    llm_eval_rows.append({
        "id": case["id"],
        "should_call_tool": should_call,
        "expected_tool": expected_tool,
        "selected_tool": selected_tool,
        "parser_exception": parser_exception,
        "arguments_parse_ok": arguments_parse_ok,
        "tool_choice_correct": tool_choice_correct,
        "arguments_schema_valid": schema_valid,
        "arguments_semantic_correct": semantic_ok,
        "end_to_end_correct": end_to_end,
    })


llm_df = pd.DataFrame(llm_eval_rows)
llm_df


In [ ]:
tool_cases_df = llm_df[
    llm_df["should_call_tool"]
]

no_tool_df = llm_df[
    ~llm_df["should_call_tool"]
]

metrics_b = {
    "tool_choice_accuracy": float(
        llm_df["tool_choice_correct"].mean()
    ),
    "arguments_parse_rate": float(
        tool_cases_df["arguments_parse_ok"].mean()
    ),
    "arguments_schema_valid_rate": float(
        tool_cases_df["arguments_schema_valid"]
        .astype(bool)
        .mean()
    ),
    "arguments_semantic_accuracy": float(
        tool_cases_df["arguments_semantic_correct"]
        .astype(bool)
        .mean()
    ),
    "no_tool_accuracy": float(
        no_tool_df["tool_choice_correct"].mean()
    ),
    "end_to_end_task_accuracy": float(
        llm_df["end_to_end_correct"].mean()
    ),
}

pd.DataFrame([metrics_b])


#### **17. Análisis de errores (error analysis)**

```text
error de selección de herramienta (`tool selection error`)
error de análisis sintáctico (`parse error`)
error de esquema (`error de esquema (`schema error`)`)
error semántico en argumentos (`semantic argument error`)
```

deben poder distinguirse.

In [ ]:
llm_df[
    ~llm_df["end_to_end_correct"]
]


#### **18. Manifiesto reproducible**

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return h.hexdigest()


manifest = {
    "week": 6,
    "seed": SEED,
    "run_real_tool_llm": RUN_REAL_TOOL_LLM,
    "real_model_id": (
        REAL_MODEL_ID
        if RUN_REAL_TOOL_LLM
        else None
    ),
    "python": platform.python_version(),
    "pydantic": pydantic.__version__,
    "hashes": {
        "catalogo_tools": sha256_file(
            DATA_DIR / "catalogo_tools.json"
        ),
        "benchmark_runtime": sha256_file(
            DATA_DIR / "benchmark_runtime.jsonl"
        ),
        "benchmark_tool_calls": sha256_file(
            DATA_DIR / "benchmark_tool_calls.jsonl"
        ),
        "tool_call_fixtures": sha256_file(
            DATA_DIR / "tool_call_fixtures.json"
        ),
    },
    "fault_schedule": {
        case["id"]: case["fault"]
        for case in runtime_cases
    },
    "policies": [
        {
            **policy.__dict__,
            "backoff_ms": list(policy.backoff_ms),
        }
        for policy in POLICIES
    ],
    "metrics_experiment_a": (
        metrics_a_df.to_dict(orient="records")
    ),
    "metrics_experiment_b": metrics_b,
    "limitations": [
        (
            "El Experimento A usa herramientas locales "
            "y fallos sintéticos reproducibles."
        ),
        (
            "El benchmark runtime contiene 17 casos y "
            "no demuestra generalización."
        ),
        (
            "Los fixtures offline no miden comportamiento "
            "de un LLM real."
        ),
        (
            "No se estudian agentes, ReAct, MCP runtime, "
            "parallel tool calling ni repair loops."
        ),
    ],
}

manifest_path = RESULTS_DIR / "latest_run.json"

manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Manifiesto:", manifest_path)


#### **19. Ejercicios de código**

Los ejercicios siguientes practican los mecanismos centrales de la semana.

Reglas:

```text
- no utilizar LangChain, LangGraph ni agentes,
- no introducir un segundo LLM,
- conservar 0 o 1 llamada de herramienta por solicitud,
- diferenciar errores de protocolo, validación, dominio y ejecución,
- no usar `except Exception` para decidir reintentos,
- mantener pruebas deterministas.
```

Los esqueletos pueden ejecutarse sin modificar el resto del cuaderno. Las pruebas sugeridas permanecen comentadas para que el cuaderno canónico continúe ejecutándose aunque los ejercicios todavía no estén resueltos.

##### **Ejercicio 1 - Lista de herramientas permitidas (`allowlist`)**

Implementa `validate_tool_name()`.

Debes:

```text
1. aceptar un nombre de herramienta,
2. devolver el mismo nombre si pertenece a TOOL_IMPLS,
3. devolver un ToolResult con error `protocol` si la herramienta no existe,
4. no ejecutar ninguna herramienta.
```

Pregunta:

> ¿Por qué esta validación debe ocurrir antes de validar los argumentos?.

In [ ]:
def validate_tool_name(
    tool_name: str,
) -> str | ToolResult:
    # TODO: validar contra TOOL_IMPLS.
    return tool_name


# Pruebas sugeridas:
# assert validate_tool_name("catalog.lookup_stock") == "catalog.lookup_stock"
# result = validate_tool_name("catalog.delete_product")
# assert isinstance(result, ToolResult)
# assert result.status == "error"
# assert result.error.kind == "protocol"

##### **Ejercicio 2 - Validación estructural frente a validación de dominio**

Implementa `validate_stock_request()`.

Casos esperados:

```text
"SKU001" -> válido estructuralmente y válido para el dominio
"SKU999" -> válido estructuralmente e inválido para el dominio
{}       -> inválido estructuralmente
```

La función debe devolver `ToolResult(status="ok", ...)` o un error con `kind="validation"` o `kind="domain"`.

No utilices un LLM para decidir si el SKU existe.

In [ ]:
def validate_stock_request(
    raw_arguments: dict[str, Any],
) -> ToolResult:
    # TODO:
    # 1. validar con LookupStockInput,
    # 2. si Pydantic falla -> validation,
    # 3. si el SKU no existe -> domain,
    # 4. si todo es correcto -> ok.
    return ToolResult(status="ok", data=raw_arguments)


# Pruebas sugeridas:
# assert validate_stock_request({"sku": "SKU001"}).status == "ok"
# assert validate_stock_request({"sku": "SKU999"}).error.kind == "domain"
# assert validate_stock_request({}).error.kind == "validation"

##### **Ejercicio 3 - Clasificación de excepciones**

Implementa `exception_to_tool_error()` para convertir excepciones internas en un contrato estable.

Reglas:

```text
DomainError            -> domain     -> retryable = False
TransientToolError     -> transient  -> retryable = True
asyncio.TimeoutError   -> timeout    -> retryable = True
otra excepción         -> execution  -> retryable = False
```

No utilices el texto del mensaje para decidir la clase de error.

In [ ]:
def exception_to_tool_error(
    exc: Exception,
) -> ToolError:
    # TODO: clasificar por tipo de excepción.
    return ToolError(
        code="not_implemented",
        kind="execution",
        retryable=False,
        message=str(exc),
    )


# Pruebas sugeridas:
# assert exception_to_tool_error(DomainError("x")).kind == "domain"
# assert exception_to_tool_error(TransientToolError("x")).retryable is True
# assert exception_to_tool_error(asyncio.TimeoutError()).kind == "timeout"

##### **Ejercicio 4 - Política de reintento selectivo (`retry`)**

Implementa `should_retry()`.

Debes devolver `True` solamente cuando:

```text
error.retryable == True
y
attempt < max_attempts
```

En la política de esta semana, los errores `protocol`, `validation`, `domain` y `output_contract` no se reintentan automáticamente.

Pregunta:

> ¿Por qué repetir exactamente los mismos argumentos no repara un error de validación?.

In [ ]:
def should_retry(
    error: ToolError,
    attempt: int,
    max_attempts: int,
) -> bool:
    # TODO: implementar la política.
    return False


# Pruebas sugeridas:
# transient = ToolError(
#     code="temporary_unavailable",
#     kind="transient",
#     retryable=True,
#     message="temporal",
# )
# domain = ToolError(
#     code="domain_error",
#     kind="domain",
#     retryable=False,
#     message="no existe",
# )
# assert should_retry(transient, 1, 3) is True
# assert should_retry(transient, 3, 3) is False
# assert should_retry(domain, 1, 3) is False

##### **Ejercicio 5 - Presupuesto temporal restante**

Implementa `remaining_budget_ms()` y `can_backoff()`.

`remaining_budget_ms()` recibe el instante inicial y el límite temporal total (`overall_deadline_ms`) y devuelve los milisegundos restantes. Nunca debes devolver un valor negativo.

`can_backoff()` responde si queda presupuesto suficiente para realizar la espera entre reintentos (`backoff_ms`) antes de otro intento.

No uses `sleep()` dentro de estas funciones.

In [ ]:
def remaining_budget_ms(
    started: float,
    overall_deadline_ms: float,
) -> float:
    # TODO: descontar el tiempo transcurrido.
    return overall_deadline_ms


def can_backoff(
    remaining_ms: float,
    backoff_ms: float,
) -> bool:
    # TODO: decidir si la espera cabe en el presupuesto restante.
    return False


# Prueba sugerida:
# started = time.perf_counter()
# await asyncio.sleep(0.02)
# remaining = remaining_budget_ms(started, overall_deadline_ms=100)
# assert 0 <= remaining < 100
# assert can_backoff(remaining, 10) == (remaining > 10)

##### **Ejercicio 6 - Validación del contrato de salida**

Implemente `validate_tool_output()`.

Debe:

```text
1. seleccionar el modelo usando OUTPUT_MODELS,
2. validar raw_output,
3. devolver ToolResult(status="ok") si cumple el contrato,
4. devolver error `output_contract` si no lo cumple.
```

La salida:

```python
{"route": "LIM-CUS", "cost": "unknown"}
```

no debe aceptarse como resultado de `shipping.get_quote`.

In [ ]:
def validate_tool_output(
    tool_name: str,
    raw_output: dict[str, Any],
) -> ToolResult:
    # TODO: validar usando OUTPUT_MODELS.
    return ToolResult(status="ok", data=raw_output)


# Pruebas sugeridas:
# valid = validate_tool_output(
#     "shipping.get_quote",
#     {
#         "origin": "LIM",
#         "destination": "CUS",
#         "weight_kg": 2.0,
#         "price": 15.0,
#     },
# )
# invalid = validate_tool_output(
#     "shipping.get_quote",
#     {"route": "LIM-CUS", "cost": "unknown"},
# )
# assert valid.status == "ok"
# assert invalid.error.kind == "output_contract"

##### **Ejercicio 7 - Evaluación de una llamada de herramienta (`ToolCall`)**

Implementa `evaluate_tool_call()` para un caso del Experimento B.

Debes informar:

```text
tool_choice_correct
arguments_schema_valid
arguments_semantic_correct
```

Supone que `call` ya fue analizada por el parser. No ejecutes la herramienta.

Ejemplo:

```text
esperado: catalog.lookup_stock {"sku": "SKU003"}
producido: catalog.lookup_stock {"sku": "SKU002"}
```

Resultado esperado:

```text
tool_choice_correct        = True
arguments_schema_valid     = True
arguments_semantic_correct = False
```

Este ejercicio fuerza la distinción:

```text
llamada válida != llamada correcta
```

In [ ]:
def evaluate_tool_call(
    expected_tool: str,
    expected_arguments: dict[str, Any],
    call: dict[str, Any] | None,
) -> dict[str, bool]:
    # TODO: utilizar INPUT_MODELS, normalize_tool_call
    # y semantic_arguments_match.
    return {
        "tool_choice_correct": False,
        "arguments_schema_valid": False,
        "arguments_semantic_correct": False,
    }


# Prueba sugerida:
# result = evaluate_tool_call(
#     expected_tool="catalog.lookup_stock",
#     expected_arguments={"sku": "SKU003"},
#     call={
#         "name": "catalog.lookup_stock",
#         "arguments": {"sku": "SKU002"},
#     },
# )
# assert result == {
#     "tool_choice_correct": True,
#     "arguments_schema_valid": True,
#     "arguments_semantic_correct": False,
# }

#### **20 . Criterio de cierre**

Debes poder explicar sin depender del notebook:

```text
salida estructurada (`structured output`) != llamada de herramientas (`tool calling`)
llamada de herramientas (`tool calling`) != ejecución de herramientas (`tool execution`)
especificación de herramienta (`tool specification`) != implementación
válido por esquema (`schema-valid`) != válido para el dominio (`domain-valid`)
LLM output != trusted input
excepción (`exception`) != `ToolError`
reintento (`retry`) != reparación (`repair`)
tiempo de espera (`timeout`) por intento != límite temporal total (`deadline`)
llamada a herramienta válida (`tool call`) != llamada a herramienta correcta
llamada de funciones (`function calling`) != agente
MCP != llamada de funciones (`function calling`)